# Treinamento — modelo de vocals (Google Colab)

Notebook Colab: treina `VocalsCRNN` com dataset no Google Drive,
avalia no val set e oferece o download do modelo treinado.

**Não inclui** o teste end-to-end de chart/preview — rode `main.py`
localmente depois de baixar o `.pt`.


## 0. Setup (execute apenas uma vez por sessão)

In [ ]:
# Instala dependências
%pip install -q librosa mido openpyxl tqdm

# Clona o repositório Playsic (contém os módulos de treinamento)
import os, sys
if not os.path.exists('/content/playsic'):
    os.system('git clone https://github.com/ArturXCX/playsic_prototype.git /content/playsic')
    print("Repositório clonado.")
else:
    print("Repositório já presente.")

sys.path.insert(0, '/content/playsic')
sys.path.insert(0, '/content/playsic/treinamento')
sys.path.insert(0, '/content/playsic/processamento/midi_excel')

# Monta o Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Drive montado.")


## 1. Verificação de GPU

In [ ]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM  : {props.total_memory / 1e9:.1f} GB")
else:
    print("\n⚠  GPU não encontrada!")
    print("   Vá em: Ambiente de execução → Alterar tipo de ambiente de execução → GPU T4")


## 2. Paths e hiperparâmetros

In [ ]:
from pathlib import Path
import random, numpy as np

# ── Ajuste se o nome da sua pasta no Drive for diferente ────────────────────
DRIVE_ROOT  = Path('/content/drive/MyDrive/playsic')  # ← pasta raiz no Drive
DATASET_DIR = DRIVE_ROOT / 'dataset'                   # ← onde está o dataset
CKPT_DIR    = DRIVE_ROOT / 'checkpoint' / 'vocals'      # ← onde salvar o modelo
CKPT_DIR.mkdir(parents=True, exist_ok=True)

CKPT_PATH   = CKPT_DIR / 'vocals_crnn_best.pt'
META_PATH   = CKPT_DIR / 'vocals_crnn_meta.pt'

assert DATASET_DIR.is_dir(), (
    f"Dataset não encontrado: {DATASET_DIR}\n"
    "Ajuste DRIVE_ROOT acima para corresponder ao nome da sua pasta no Drive."
)

# ── Hiperparâmetros ──────────────────────────────────────────────────────────
SEED           = 42
N_MELS         = 128
BATCH_SIZE     = 32    # T4 16 GB aguenta; reduza para 16 se der OOM
EPOCHS         = 80
LR             = 1e-3
WEIGHT_DECAY   = 1e-4
GRAD_CLIP      = 1.0
PATIENCE       = 15
POS_WEIGHT_CAP = 50.0
VAL_FRAC       = 0.15
MIN_NOTE_COUNT = 10

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f"Dataset : {DATASET_DIR}")
print(f"Ckpt    : {CKPT_PATH}")
print(f"Meta    : {META_PATH}")


## 3. Imports

In [ ]:
import math
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import torch.nn as nn
from torch.utils.data import DataLoader

from vocals_crnn import (
    VocalsCRNN, LANES, LANE_NAMES, N_LANES, AUDIO_STEM,
    VOCAL_MIDI_MIN, VOCAL_MIDI_MAX, count_params,
)
from audio_features import (
    SAMPLE_RATE, SUBDIV_PER_BEAT, TICKS_PER_BEAT,
    audio_to_grid_mel, normalize_mel, step_duration_seconds,
)
from notes_xlsx import parse_vocals_events, predictions_to_xlsx
from training_utils import (
    SongData, list_song_dirs, preprocess_song,
    compute_mel_stats, DrumChartDataset, CHUNK_STEPS,
)
print(f"Vocal pitch range filtrado: MIDI {VOCAL_MIDI_MIN} – {VOCAL_MIDI_MAX}")
print(f"N_LANES = {N_LANES}  ({LANE_NAMES})")


## 4. Pré-processamento do dataset

In [ ]:
print(f"Procurando músicas em: {DATASET_DIR}")
song_dirs = list_song_dirs(DATASET_DIR, audio_stem=AUDIO_STEM)
print(f"Encontradas {len(song_dirs)} músicas com "vocals.ogg" + notes.xlsx")
assert len(song_dirs) > 0, "Nenhuma música encontrada — verifique DATASET_DIR"

print("Carregando + extraindo mel-espectrograma...")
songs, skipped_none, skipped_empty = [], 0, 0
for d in tqdm(song_dirs):
    s = preprocess_song(d, lanes_map=LANES, n_lanes=N_LANES,
                        n_mels=N_MELS, audio_stem=AUDIO_STEM,
                        parse_events_fn=parse_vocals_events)
    if s is None:
        skipped_none += 1
        continue
    if s.target.sum() < MIN_NOTE_COUNT:
        skipped_empty += 1   # sem vocals no range
        continue
    songs.append(s)

print(f"OK: {len(songs)} músicas pré-processadas")
print(f"   Puladas por erro:        {skipped_none}")
print(f"   Puladas (sem vocals no range): {skipped_empty}")
assert len(songs) > 0, "Nenhuma música útil — verifique o dataset"


## 5. Split + Dataset + Loaders

In [ ]:
random.shuffle(songs)
n_val = max(1, int(len(songs) * VAL_FRAC))
val_songs   = songs[:n_val]
train_songs = songs[n_val:]
print(f"Train: {len(train_songs)}   Val: {len(val_songs)}")

mel_mean, mel_std = compute_mel_stats(train_songs)
print(f"Mel stats: mean={mel_mean:.3f}  std={mel_std:.3f}")

train_ds = DrumChartDataset(train_songs, mel_mean, mel_std, augment=True)
val_ds   = DrumChartDataset(val_songs,   mel_mean, mel_std, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
print(f"Batches — train: {len(train_loader)}  val: {len(val_loader)}")


## 6. Modelo + pos_weight

In [ ]:
def compute_pos_weight(songs):
    pos = torch.zeros(N_LANES)
    neg = torch.zeros(N_LANES)
    for s in songs:
        pos += s.target.sum(dim=0)
        neg += (1 - s.target).sum(dim=0)
    pw = (neg / pos.clamp(min=1)).clamp(max=POS_WEIGHT_CAP)
    print({n: round(v, 2) for n, v in zip(LANE_NAMES, pw.tolist())})
    return pw

model = VocalsCRNN(n_mels=N_MELS).to(DEVICE)
print(f"Parâmetros: {count_params(model):,}")

pos_weight = compute_pos_weight(train_songs)


## 7. Treino com early stopping

Loss: BCE + pos_weight. Mixed precision. Checkpoint pelo melhor F1 macro.

In [ ]:
import collections

@torch.no_grad()
def compute_metrics(probs, targets, threshold=0.5):
    preds = (probs >= threshold).float()
    out = {}
    f1s = []
    for lane in range(N_LANES):
        p = preds[..., lane].flatten()
        t = targets[..., lane].flatten()
        tp = ((p == 1) & (t == 1)).sum().item()
        fp = ((p == 1) & (t == 0)).sum().item()
        fn = ((p == 0) & (t == 1)).sum().item()
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        out[f"f1_{LANE_NAMES[lane]}"] = f1
        f1s.append(f1)
    out["f1_macro"] = float(np.mean(f1s))

    pred_cnt = preds.sum(dim=-1).flatten()
    true_cnt = targets.sum(dim=-1).flatten()
    pred_has = (pred_cnt >= 1).float()
    true_has = (true_cnt >= 1).float()
    tp = ((pred_has == 1) & (true_has == 1)).sum().item()
    fp = ((pred_has == 1) & (true_has == 0)).sum().item()
    fn = ((pred_has == 0) & (true_has == 1)).sum().item()
    po = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    ro = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    out["onset_f1"]   = 2 * po * ro / (po + ro) if (po + ro) > 0 else 0.0
    out["onset_prec"] = po
    out["onset_rec"]  = ro
    out["count_acc"]  = (pred_cnt == true_cnt).float().mean().item()
    out["count_mae"]  = (pred_cnt - true_cnt).abs().float().mean().item()
    return out


def train_one_epoch(model, loader, optim, criterion, scaler):
    model.train()
    losses = []
    for mel, target in loader:
        mel    = mel.to(DEVICE, non_blocking=True)
        target = target.to(DEVICE, non_blocking=True)
        optim.zero_grad()
        with torch.cuda.amp.autocast(enabled=DEVICE.type == "cuda"):
            logits = model(mel)
            loss   = criterion(logits, target)
        scaler.scale(loss).backward()
        scaler.unscale_(optim)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optim)
        scaler.update()
        losses.append(loss.item())
    return float(np.mean(losses))


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    losses, P, T = [], [], []
    for mel, target in loader:
        mel    = mel.to(DEVICE)
        target = target.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=DEVICE.type == "cuda"):
            logits = model(mel)
            loss   = criterion(logits, target)
        losses.append(loss.item())
        P.append(torch.sigmoid(logits).cpu())
        T.append(target.cpu())
    probs = torch.cat(P, dim=0)
    tgts  = torch.cat(T, dim=0)
    out   = compute_metrics(probs, tgts)
    out["loss"] = float(np.mean(losses))
    return out


criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))
optim_    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched     = torch.optim.lr_scheduler.CosineAnnealingLR(optim_, T_max=EPOCHS)
scaler    = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")

best_f1, no_improve, history = -1.0, 0, []
for epoch in range(1, EPOCHS + 1):
    tl = train_one_epoch(model, train_loader, optim_, criterion, scaler)
    v  = evaluate(model, val_loader, criterion)
    sched.step()
    history.append({"epoch": epoch, "train_loss": tl, **v})

    print(f"[ep {epoch:3d}] tr={tl:.4f} vl={v['loss']:.4f} | "
          f"F1m={v['f1_macro']:.3f} ons={v['onset_f1']:.3f} cnt={v['count_acc']:.3f} | "
          f"V={v['f1_VocalActive']:.3f}")

    if v["f1_macro"] > best_f1:
        best_f1, no_improve = v["f1_macro"], 0
        torch.save({"model": model.state_dict(),
                    "epoch": epoch,
                    "val_f1_macro": best_f1}, CKPT_PATH)
        print(f"   → checkpoint salvo em Drive (F1m={best_f1:.4f})")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping (sem melhora por {PATIENCE} epochs)")
            break

torch.save({"mel_mean": mel_mean, "mel_std": mel_std,
            "history": history, "n_mels": N_MELS}, META_PATH)
print(f"Treino concluído.  Best F1_macro = {best_f1:.4f}")
print(f"Checkpoint: {CKPT_PATH}")
print(f"Meta      : {META_PATH}")


## 8. Threshold tuning por lane

Encontra o melhor threshold no val set e salva no meta.

In [ ]:
@torch.no_grad()
def collect_val_predictions(model, val_songs):
    model.eval()
    P, T = [], []
    for s in val_songs:
        mel   = normalize_mel(s.mel, mel_mean, mel_std).unsqueeze(0).to(DEVICE)
        probs = torch.sigmoid(model(mel)).squeeze(0).cpu().numpy()
        P.append(probs)
        T.append(s.target.numpy())
    return np.concatenate(P, 0), np.concatenate(T, 0)


def f1_at(probs, targets, lane, t):
    p  = (probs[:, lane] >= t).astype(np.int32)
    g  = targets[:, lane].astype(np.int32)
    tp = int(((p == 1) & (g == 1)).sum())
    fp = int(((p == 1) & (g == 0)).sum())
    fn = int(((p == 0) & (g == 1)).sum())
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return f1, prec, rec


ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model"])
print(f"Carregado: epoch {ckpt['epoch']}  F1_macro={ckpt['val_f1_macro']:.4f}")

probs, targets = collect_val_predictions(model, val_songs)
thresholds = np.arange(0.05, 0.95, 0.025)

optimal_thresholds = {}
print(f"\n{'Lane':<12s} {'thr':>6s} {'F1':>6s} {'P':>6s} {'R':>6s}")
print("─" * 42)
for lane in range(N_LANES):
    best = max(((t, *f1_at(probs, targets, lane, t)) for t in thresholds),
               key=lambda x: x[1])
    t, f1, p, r = best
    optimal_thresholds[LANE_NAMES[lane]] = float(t)
    print(f"{LANE_NAMES[lane]:<12s} {t:>6.3f} {f1:>6.3f} {p:>6.3f} {r:>6.3f}")

meta = torch.load(META_PATH, map_location="cpu")
meta["optimal_thresholds"] = optimal_thresholds
torch.save(meta, META_PATH)
print(f"\nThresholds salvos em {META_PATH}")


## 9. Métricas comparativas: estrito × onset × contagem
> **Nota vocals v1**: com `N_LANES=1` o modelo prevê apenas "tem vocal ativo?", logo `estrito ≡ onset`. A contagem ainda dá informação útil (densidade vocal).

- **estrito**  — F1 macro com thresholds otimizados. Objetivo principal.
- **onset**    — "previu nota onde tem nota?" — F1 binário sobre `sum > 0`.
- **contagem** — % de steps com número exato; MAE da diferença.


In [ ]:
thr_arr   = np.array([optimal_thresholds[LANE_NAMES[i]] for i in range(N_LANES)])
preds_opt = (probs >= thr_arr[None, :]).astype(np.float32)
gt        = targets.astype(np.float32)

# Estrito
f1s_strict = []
for lane in range(N_LANES):
    f1, _, _ = f1_at(probs, gt, lane, optimal_thresholds[LANE_NAMES[lane]])
    f1s_strict.append(f1)
f1_strict = float(np.mean(f1s_strict))

# Onset
pred_cnt = preds_opt.sum(axis=1)
true_cnt = gt.sum(axis=1)
pred_has = (pred_cnt >= 1).astype(np.float32)
true_has = (true_cnt >= 1).astype(np.float32)
tp = float(((pred_has == 1) & (true_has == 1)).sum())
fp = float(((pred_has == 1) & (true_has == 0)).sum())
fn = float(((pred_has == 0) & (true_has == 1)).sum())
prec_o   = tp / (tp + fp) if (tp + fp) > 0 else 0.0
rec_o    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_onset = 2 * prec_o * rec_o / (prec_o + rec_o) if (prec_o + rec_o) > 0 else 0.0

# Contagem
count_acc = float((pred_cnt == true_cnt).mean())
count_mae = float(np.abs(pred_cnt - true_cnt).mean())

diff = (pred_cnt - true_cnt).astype(int)
dist = collections.Counter(diff.tolist())
n_total = len(diff)

print(f"\n{'='*56}")
print(f"  Métricas finais com thresholds otimizados")
print(f"{'='*56}")
print(f"  estrito   F1_macro = {f1_strict:.4f}")
print(f"  onset     F1       = {f1_onset:.4f}  (P={prec_o:.3f}  R={rec_o:.3f})")
print(f"  contagem  acurácia = {count_acc:.4f}  MAE = {count_mae:.3f}")

print(f"\n  Distribuição (pred_count - true_count):")
for d in sorted(dist.keys()):
    n   = dist[d]
    pct = 100.0 * n / n_total
    bar = "█" * int(50 * n / n_total)
    print(f"    {d:+3d}: {n:7d} steps ({pct:5.1f}%) {bar}")

meta = torch.load(META_PATH, map_location="cpu")
meta["final_metrics"] = {
    "f1_strict": f1_strict,   "f1_onset":   f1_onset,
    "onset_prec": prec_o,     "onset_rec":  rec_o,
    "count_acc":  count_acc,  "count_mae":  count_mae,
    "count_diff_distribution": {int(k): int(v) for k, v in dist.items()},
}
torch.save(meta, META_PATH)
print(f"\nMétricas salvas em {META_PATH}")


## 10. Curvas de treino + piano roll

In [ ]:
def plot_training_curves(history):
    epochs = [h["epoch"] for h in history]
    fig, ax = plt.subplots(1, 3, figsize=(18, 4))

    ax[0].plot(epochs, [h["train_loss"] for h in history], label="train")
    ax[0].plot(epochs, [h["loss"]       for h in history], label="val")
    ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend()
    ax[0].set_title("Loss"); ax[0].grid(alpha=0.3)

    for lane_name in LANE_NAMES:
        ax[1].plot(epochs, [h[f"f1_{lane_name}"] for h in history],
                   label=lane_name, alpha=0.7)
    ax[1].plot(epochs, [h["f1_macro"] for h in history], "k--", lw=2, label="macro")
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("F1"); ax[1].set_ylim(0, 1)
    ax[1].set_title("F1 estrito por lane"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)

    ax[2].plot(epochs, [h["f1_macro"]  for h in history], "k-", lw=2, label="estrito (macro)")
    ax[2].plot(epochs, [h["onset_f1"]  for h in history], "g-", lw=2, label="onset")
    ax[2].plot(epochs, [h["count_acc"] for h in history], "b-", lw=2, label="count_acc")
    ax[2].set_xlabel("epoch"); ax[2].set_ylim(0, 1)
    ax[2].set_title("Estrito vs onset vs contagem"); ax[2].legend(); ax[2].grid(alpha=0.3)

    plt.tight_layout(); plt.show()


@torch.no_grad()
def piano_roll(song: SongData, max_steps: int = 400):
    mel   = normalize_mel(song.mel, mel_mean, mel_std).unsqueeze(0).to(DEVICE)
    probs = torch.sigmoid(model(mel)).squeeze(0).cpu().numpy()
    thr   = np.array([optimal_thresholds[LANE_NAMES[i]] for i in range(N_LANES)])
    preds = (probs >= thr[None, :]).astype(np.float32)
    target = song.target.numpy()
    n = min(target.shape[0], probs.shape[0], max_steps)
    fig, ax = plt.subplots(3, 1, figsize=(14, 6), sharex=True)
    ax[0].imshow(target[:n].T, aspect="auto", origin="lower", cmap="Greys", vmin=0, vmax=1)
    ax[0].set_title(f"Alvo — {song.song_id}")
    ax[0].set_yticks(range(N_LANES)); ax[0].set_yticklabels(LANE_NAMES)
    ax[1].imshow(probs[:n].T, aspect="auto", origin="lower", cmap="viridis", vmin=0, vmax=1)
    ax[1].set_title("Probabilidades preditas")
    ax[1].set_yticks(range(N_LANES)); ax[1].set_yticklabels(LANE_NAMES)
    ax[2].imshow(preds[:n].T, aspect="auto", origin="lower", cmap="Greys", vmin=0, vmax=1)
    ax[2].set_title("Predições (thresholds otimizados)")
    ax[2].set_yticks(range(N_LANES)); ax[2].set_yticklabels(LANE_NAMES)
    ax[2].set_xlabel("step (semicolcheia)")
    plt.tight_layout(); plt.show()


plot_training_curves(history)
piano_roll(val_songs[0])


## 11. Download do modelo treinado

O modelo já foi salvo no Drive em `CKPT_DIR`. As células abaixo também
iniciam o download direto para o seu computador — útil se quiser copiar
os arquivos direto para `treinamento/checkpoint/vocals/` sem abrir o Drive.


In [ ]:
from google.colab import files

print(f"Baixando  {CKPT_PATH.name} ...")
files.download(str(CKPT_PATH))

print(f"Baixando  {META_PATH.name} ...")
files.download(str(META_PATH))

print()
print("Feito! Coloque os arquivos em:")
print(f"  treinamento/checkpoint/{CKPT_DIR.name}/")
print("e rode  python main.py --audio <musica.mp3>  para testar o pipeline.")
